In [ ]:
# === FIGURE 1f RECREATION USING Session API ===

%load_ext autoreload
%autoreload 2
%matplotlib inline

from rl_analysis.plotting import savefig, setup_plotting_env, clean_ticks, geno_palette
from rl_analysis.photometry.snippet import get_shuffle_average

from functools import partial
from tqdm.auto import tqdm
from joblib import Parallel, delayed

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import seaborn as sns
import toml
import multiprocessing as mp

from scipy import signal
from scipy.ndimage import uniform_filter1d
from scipy.signal import find_peaks
from scipy.stats import wilcoxon
from numba import jit

# ---------------------------------------------------------------------
# Load analysis configuration (paths for saving figures / stats)
# ---------------------------------------------------------------------
with open("../analysis_configuration.toml", "r") as f:
    analysis_config = toml.load(f)

mp.set_start_method("spawn", force=True)

raw_dirs = analysis_config.get("raw_data", {})
proc_dirs = analysis_config.get("intermediate_results", {})
dlight_cfg = analysis_config.get("dlight_basic_analysis", {})
dlight_common_cfg = analysis_config.get("dlight_common", {})
figure_cfg = analysis_config.get("figures", {})
common_cfg = analysis_config.get("common", {})

# ---------------------------------------------------------------------
# Global constants for THIS notebook
# ---------------------------------------------------------------------
fs = float(common_cfg.get("fs", 80.0))  # sampling rate (Hz); your data is 80 Hz
mm_per_px = 1.0                          # 1 px = 1 mm as per your note

renormalize = True
dask_address = None

# brain region to use ("dms" vs "dls", etc.; case-insensitive)
use_area = "dms"

# column key for downstream analysis
use_key = "signal_reref_dff_z_renorm" if renormalize else "signal_reref_dff_z"

# long window for snippet extraction (seconds, around onset)
long_window_seconds = (-10.0, 10.0)

# window to KEEP after renormalization, in seconds (like pre_trim in original)
pre_trim_seconds = (-3.0, +3.0)
pre_trim = (int(pre_trim_seconds[0] * fs), int(pre_trim_seconds[1] * fs))

# scalars used in this analysis (no vertical component for your data)
scalars = [
    "velocity_2d_mm",
    "velocity_angle",
    "acceleration_2d_mm",
    "jerk_2d_mm",
]

# columns the original notebook expects in snippet_df
load_keys = [
    "x",
    "session_number",
    "uuid",
    "signal_reref_dff_z",
    "mouse_id",
    "snippet",
    "duration",
    "area",
    "opsin",
    "target_syllable",
    "signal_max",
    "signal_reference_corr",
    "stim_duration",
    "syllable",
] + scalars

# ---------------------------------------------------------------------
# Helper functions: kinematics, dLight column, event detection, snippets
# ---------------------------------------------------------------------
def ensure_planar_kinematics(df, fs, mm_per_px=1.0):
    """
    Make sure df has:
      - centroid_x_mm, centroid_y_mm
      - velocity_2d_mm
      - acceleration_2d_mm
      - jerk_2d_mm
      - velocity_angle

    Assumes centroid_x / centroid_y exist (in pixels).
    """
    df = df.copy()

    if "centroid_x_mm" not in df.columns and "centroid_x" in df.columns:
        df["centroid_x_mm"] = df["centroid_x"] * mm_per_px
        df["centroid_y_mm"] = df["centroid_y"] * mm_per_px

    # 2D velocity (mm/s)
    if "velocity_2d_mm" not in df.columns:
        x = df["centroid_x_mm"].to_numpy()
        y = df["centroid_y_mm"].to_numpy()
        dx = np.gradient(x)
        dy = np.gradient(y)
        df["velocity_2d_mm"] = np.sqrt(dx**2 + dy**2) * fs

    # 2D acceleration (mm/s^2)
    if "acceleration_2d_mm" not in df.columns:
        v = df["velocity_2d_mm"].to_numpy()
        dv = np.gradient(v)
        df["acceleration_2d_mm"] = dv * fs

    # 2D jerk (mm/s^3)
    if "jerk_2d_mm" not in df.columns:
        a = df["acceleration_2d_mm"].to_numpy()
        da = np.gradient(a)
        df["jerk_2d_mm"] = da * fs

    # velocity angle (radians)
    if "velocity_angle" not in df.columns:
        x = df["centroid_x_mm"].to_numpy()
        y = df["centroid_y_mm"].to_numpy()
        dx = np.gradient(x)
        dy = np.gradient(y)
        vx = dx * fs
        vy = dy * fs
        df["velocity_angle"] = np.arctan2(vy, vx)

    return df


def detect_syllable_onsets_positions(df, syllable_col="syllable"):
    """
    Return integer positions (iloc indices) of syllable onsets
    defined as changes in the 'syllable' column to a non-NA value.
    """
    if syllable_col not in df.columns:
        return np.array([], dtype=int)
    syl = df[syllable_col]
    change = syl.ne(syl.shift(1)) & syl.notna()
    return np.where(change.to_numpy())[0]


def detect_movement_initiations_positions(df, fs, vel_col="velocity_2d_mm"):
    """
    Markowitz 2022 Methods:
      - z-score 2D velocity
      - smooth with a 50-point boxcar at 30 Hz (~1.67 s) -> we match 1.67 s in time
      - take derivative of smoothed trace
      - cube derivative (changepoint score)
      - find peaks with height=1, width=1, prominence=1
    Returns integer positions (iloc indices) of detected peaks.
    """
    if vel_col not in df.columns:
        return np.array([], dtype=int)

    v = df[vel_col].to_numpy().astype(float)
    if np.all(np.isnan(v)):
        return np.array([], dtype=int)

    # z-score across experiment
    v_z = (v - np.nanmean(v)) / np.nanstd(v)

    # smooth with 1.67 s boxcar (same time window as 50 samples @ 30 Hz)
    win_samples = int(round(1.67 * fs))
    win_samples = max(win_samples, 1)
    v_smooth = uniform_filter1d(v_z, size=win_samples, mode="nearest")

    # derivative in sample space (no explicit / dt in original Methods)
    dv = np.diff(v_smooth, prepend=v_smooth[0])

    # changepoint score
    cp_score = dv ** 3

    # detect peaks
    peaks, props = find_peaks(cp_score, height=1, width=1, prominence=1)
    return peaks.astype(int)


def get_dlight_column_for_area(s, area, z_suffix="_zscore"):
    """
    Given a Session and a brain region name (e.g. 'dms'),
    return the corresponding signal_k_zscore column name.
    """
    meta = getattr(s, "metadata", {}) or {}
    region_map = meta.get("fiber_regions_by_index") or {}

    area = area.lower()
    match_idx = None
    for k, reg in region_map.items():
        if str(reg).lower() == area:
            match_idx = k
            break

    if match_idx is None:
        raise KeyError(
            f"No fiber for area '{area}' in session {getattr(s, 'session_id', '<no id>')}"
        )

    col = f"signal_{match_idx}{z_suffix}"
    if col not in s.all_data_df.columns:
        raise KeyError(
            f"Column {col} not found in all_data_df for session {getattr(s, 'session_id', '<no id>')}"
        )
    return col


def build_snippet_df_from_sessions(
    sessions,
    use_area,
    fs,
    long_window_seconds=(-10.0, 10.0),
    mm_per_px=1.0,
):
    """
    Build a snippet_df that mimics the original
    dlight_snippets_offline_longwin + dlight_snippets_movements_longwin parquet,
    but using the Session API instead.

    - For each session:
      - ensure planar kinematics (vel, accel, jerk, velocity_angle)
      - detect syllable onsets
      - detect movement initiations
      - for each event, extract a long window around the event
        (long_window_seconds, e.g. [-10, 10] s)
      - populate per-sample rows with x (time rel to event), duration, etc.
    """
    rows = []

    for session_number, s in enumerate(sessions):
        sid = getattr(s, "session_id", f"session_{session_number}")

        # Choose the correct dLight channel for the requested area
        try:
            dlight_col = get_dlight_column_for_area(s, use_area)
        except KeyError as e:
            print(f"Skipping session {sid}: {e}")
            continue

        # Work on a copy of all_data_df with 0-based integer index
        adf = s.all_data_df.reset_index(drop=True)
        adf = ensure_planar_kinematics(adf, fs, mm_per_px=mm_per_px)

        if "syllable" not in adf.columns:
            print(f"Skipping session {sid}: no 'syllable' column in all_data_df")
            continue

        uuid = sid
        meta = getattr(s, "metadata", {}) or {}
        mouse_id = meta.get("animal_id", "UNK")
        opsin = meta.get("opsin", "UNK")

        n = len(adf)

        # syllable and movement onsets (positions, not times)
        syllable_onsets_pos = detect_syllable_onsets_positions(adf)
        movement_onsets_pos = detect_movement_initiations_positions(
            adf, fs, vel_col="velocity_2d_mm"
        )

        lw_start, lw_stop = long_window_seconds

        # helper for both init types
        for init_type, onset_positions in (
            ("syllables_offline", syllable_onsets_pos),
            ("movement", movement_onsets_pos),
        ):
            for local_idx, pos in enumerate(onset_positions):
                start_pos = pos + int(lw_start * fs)
                stop_pos = pos + int(lw_stop * fs)

                # skip if window doesn't fully fit inside session
                if start_pos < 0 or stop_pos >= n:
                    continue

                snippet_id = f"{uuid}_{init_type}_{local_idx}"
                positions = np.arange(start_pos, stop_pos + 1)
                rel_t = (positions - pos) / fs
                duration_s = (stop_pos - start_pos + 1) / fs

                w = adf.iloc[start_pos : stop_pos + 1]

                for idx_row, t_rel in zip(w.index, rel_t):
                    rows.append(
                        {
                            # time / indexing
                            "x": t_rel,
                            "duration": duration_s,
                            "snippet": snippet_id,
                            "init_type": init_type,
                            "session_number": session_number,
                            "uuid": uuid,
                            # meta
                            "mouse_id": mouse_id,
                            "area": use_area.lower(),
                            "opsin": opsin,
                            "target_syllable": np.nan,
                            "signal_max": np.nan,
                            "signal_reference_corr": np.nan,
                            "stim_duration": np.nan,
                            # behavior
                            "syllable": w.loc[idx_row, "syllable"],
                            "velocity_2d_mm": w.loc[idx_row, "velocity_2d_mm"],
                            "velocity_angle": w.loc[idx_row, "velocity_angle"],
                            "acceleration_2d_mm": w.loc[idx_row, "acceleration_2d_mm"],
                            "jerk_2d_mm": w.loc[idx_row, "jerk_2d_mm"],
                            # dLight (we treat signal_k_zscore as signal_reref_dff_z)
                            "signal_reref_dff_z": w.loc[idx_row, dlight_col],
                        }
                    )

    snippet_df = pd.DataFrame.from_records(rows)
    return snippet_df


# ---------------------------------------------------------------------
# Build snippet_df from sessions (instead of reading parquets)
# ---------------------------------------------------------------------
snippet_df = build_snippet_df_from_sessions(
    sessions=sessions,
    use_area=use_area,
    fs=fs,
    long_window_seconds=long_window_seconds,
    mm_per_px=mm_per_px,
)

print(f"Built snippet_df with {len(snippet_df)} rows from sessions")

# ---------------------------------------------------------------------
# From here on, we follow the original notebook logic as closely as possible
# (including quirks), just with fs instead of hard-coded 30 Hz
# ---------------------------------------------------------------------

# Drop rows missing basic timing info
snippet_df = snippet_df.dropna(subset=["x", "duration"])
snippet_df.index = range(len(snippet_df))

# Convert 'x' and 'duration' from seconds to samples (like *30 in original)
snippet_df["x"] = (snippet_df["x"] * fs).round().astype("int")
snippet_df["duration"] = (snippet_df["duration"] * fs).round().astype("int")

# Z-scored versions of scalars, per uuid
scalars_z = [f"{_scalar}_z" for _scalar in scalars]
snippet_df[scalars_z] = snippet_df.groupby("uuid")[scalars].transform(
    lambda x: (x - x.mean()) / x.std()
)

# Keep only uuids with at least some snippet diversity
snippet_df = snippet_df.groupby("uuid").filter(lambda x: x["snippet"].nunique() > 5)

# Factorize snippet identifiers (string) into integer codes, as original does
codes = pd.factorize(
    pd._libs.lib.fast_zip(
        [snippet_df["snippet"].values, snippet_df["init_type"].values]
    )
)[0]
snippet_df["snippet"] = codes

# ---------------------------------------------------------------------
# Renormalize dLight per snippet (same zscore_nb trick as original)
# ---------------------------------------------------------------------
@jit(nopython=True)
def zscore_nb(values, index):
    return (values - np.nanmean(values)) / np.nanstd(values)


if renormalize:
    renormed_values = snippet_df.groupby("snippet")["signal_reref_dff_z"].transform(
        zscore_nb, engine="numba"
    )
    snippet_df["signal_reref_dff_z_renorm"] = renormed_values

# Trim to pre_trim window in samples
snippet_df = snippet_df.loc[
    snippet_df["x"].between(*pre_trim, inclusive="both")
].copy()

# ---------------------------------------------------------------------
# Compute scalar means within snippet and snippet-wise differences
# ---------------------------------------------------------------------
scalar_means = (
    snippet_df.loc[
        (snippet_df["x"] > 0) & (snippet_df["x"] < snippet_df["duration"])
    ]
    .groupby("snippet")[scalars_z]
    .mean()
)

uuids = snippet_df.drop_duplicates("snippet").set_index("snippet")["uuid"]
scalar_means = scalar_means.reset_index()
scalar_means["uuid"] = scalar_means["snippet"].map(uuids).astype("str")
scalar_means = scalar_means.set_index("snippet")

scalar_diffs = scalar_means.groupby("uuid")[scalars_z].diff()
scalar_diffs = scalar_diffs.reset_index()
scalar_diffs["uuid"] = scalar_diffs["snippet"].map(uuids).astype("str")

# bin scalar_diffs within each uuid into quartiles
scalar_bins = (
    scalar_diffs.groupby("uuid")[scalars_z]
    .transform(lambda x: pd.qcut(x, 4, labels=False))
    .astype("UInt8")
)
scalar_bins.columns = [f"{col}_bin" for col in scalar_bins.columns]
scalar_diffs[scalar_bins.columns] = scalar_bins

# map snippet-wise bins back to snippet_df
for col in tqdm(scalar_bins.columns):
    snippet_df[col] = snippet_df["snippet"].map(
        scalar_diffs.set_index("snippet")[col]
    )

# ---------------------------------------------------------------------
# High-pass filter dLight trace (Bessel 1 Hz high-pass, generalized to fs)
# ---------------------------------------------------------------------
tqdm.pandas()

# 1 Hz corner, normalized by Nyquist (fs/2)
corner_hz = 1.0
soscoeffs = signal.bessel(
    1,
    corner_hz / (fs / 2.0),
    btype="high",
    output="sos",
)

filter_func = lambda x: signal.sosfiltfilt(soscoeffs, x)

snippet_df["signal_reref_dff_z_hp"] = snippet_df.groupby("snippet")[use_key].progress_transform(
    filter_func
)

# ---------------------------------------------------------------------
# Feature definition and binning (same quirky logic as original)
# ---------------------------------------------------------------------
use_feature = "signal_reref_dff_z_hp"

features = (
    snippet_df.loc[snippet_df["x"].between(0.0, 0.3, inclusive="both")]
    .groupby("snippet")[use_feature]
    .agg(["max"])
    # .agg(["max", "min", "mean"])
)

# now we're going to bin dLight in each scalar bin...
for col in tqdm(scalar_bins.columns):
    use_scalar_bins = scalar_diffs.set_index("snippet")[col]
    features = features.loc[features.index.intersection(use_scalar_bins.index)]
    feature_bins = features.groupby(use_scalar_bins).transform(
        lambda x: pd.qcut(x, 4, labels=False)
    )
    for _feature in tqdm(feature_bins.columns):
        snippet_df[f"{_feature}_{col}_bin"] = (
            snippet_df["snippet"].map(feature_bins[_feature]).astype("UInt8")
        )

for _feature in tqdm(features.columns):
    feature_bins = pd.qcut(features[_feature], 4, labels=False)
    snippet_df[f"{_feature}_bin"] = (
        snippet_df["snippet"].map(feature_bins).astype("UInt8")
    )

# ---------------------------------------------------------------------
# Final bin columns and clean-up
# ---------------------------------------------------------------------
bin_cols = snippet_df.filter(regex="bin$").columns
snippet_df = snippet_df.dropna(subset=bin_cols)
for _col in bin_cols:
    snippet_df[_col] = snippet_df[_col].astype("uint8")

plt_scalars = ["velocity_2d_mm_z", "jerk_2d_mm_z"]
plt_scalar_bins = [f"{_plt_scalar}_bin" for _plt_scalar in plt_scalars]
plt_dlight_bins = [f"max_{_plt_scalar}_bin_bin" for _plt_scalar in plt_scalars]

use_df = snippet_df[
    [
        use_feature,
        "snippet",
        "x",
        "max_bin",
        "init_type",
        "mouse_id",
    ]
    + plt_scalar_bins
    + plt_dlight_bins
].copy()

groupby_arrays = [snippet_df["init_type"].values]
use_df.index = range(len(use_df))

# ---------------------------------------------------------------------
# Randomization with peak value bins (as in original)
# ---------------------------------------------------------------------
signal_key = use_feature

lags = (
    snippet_df.groupby(["x", "init_type"])[signal_key]
    .mean()
    .groupby("init_type")
    .apply(lambda x: x.idxmin()[0])
)

nshuffles = 100
snippet_lst = snippet_df["snippet"].unique()
shuffle_quantity = pd.Series(use_df["snippet"].unique()).rename("snippet")

dlight_bin = ["x", "init_type", "mouse_id"]

# time windows in samples, matching original logic but using fs
all_xlims = (-1.5 * fs, 1.5 * fs)
shuffle_xlims = pre_trim

n_jobs = 30

func = partial(
    get_shuffle_average,
    use_df=use_df[dlight_bin + [use_feature, "snippet"]].loc[
        use_df["x"].between(*all_xlims, inclusive="both")
    ],
    shuffle_quantity=shuffle_quantity,
    shuffle_x=False,
    dlight_bin=dlight_bin,
)
delays = [delayed(func)(sidx) for sidx in range(nshuffles)]
results = Parallel(n_jobs=n_jobs, verbose=10, backend="multiprocessing")(delays)
observed_ave_df = pd.concat(results, axis=0)

# shuffle CI averages (shuffle x)
func = partial(
    get_shuffle_average,
    use_df=use_df[dlight_bin + [use_feature, "snippet"]].loc[
        use_df["x"].between(*shuffle_xlims, inclusive="both")
    ],
    shuffle_quantity=None,
    shuffle_x=True,
    dlight_bin=dlight_bin,
)
delays = [delayed(func)(sidx) for sidx in range(nshuffles)]
results = Parallel(n_jobs=n_jobs, verbose=10, backend="multiprocessing")(delays)
shuffle_ci_ave_df = pd.concat(results, axis=0)

# grand average shuffles over all_xlims
func = partial(
    get_shuffle_average,
    use_df=use_df[dlight_bin + [use_feature, "snippet"]].loc[
        use_df["x"].between(*all_xlims, inclusive="both")
    ],
    shuffle_quantity=None,
    shuffle_x=True,
    dlight_bin=dlight_bin,
)
delays = [delayed(func)(sidx) for sidx in range(nshuffles)]
results = Parallel(n_jobs=n_jobs, verbose=10, backend="multiprocessing")(delays)
shuffle_grand_ave_df = pd.concat(results, axis=0)

group_keys = ["x", "init_type", "mouse_id"]
shuffle_group_keys = group_keys

observed_mean = use_df.groupby(group_keys)[signal_key].mean()
shuffle_agg = shuffle_ci_ave_df.groupby(shuffle_group_keys + ["idx"]).mean()
shuffle_mu = shuffle_agg.groupby(shuffle_group_keys)[signal_key].mean()
shuffle_sig = shuffle_agg.groupby(shuffle_group_keys)[signal_key].std()
shuffle_ave_ci = shuffle_agg.groupby(["init_type", "idx"]).mean()

_test = (
    shuffle_grand_ave_df.groupby(shuffle_group_keys + ["idx"])[signal_key].mean()
    - shuffle_mu
) / shuffle_sig
_test = _test.dropna()

shuffle_cis = _test.groupby("init_type").quantile([0.025, 0.975])
shuffle_cis.index = shuffle_cis.index.set_names("ci", level=-1)
lims = shuffle_cis.groupby(["init_type", "ci"]).mean()
lims = lims.groupby("ci").apply(lambda x: x.values[np.argmax(x.values)])

observed_z = (observed_mean - shuffle_mu) / shuffle_sig
observed_z = observed_z.reset_index()
observed_z["x"] /= fs  # convert samples back to seconds for plotting

observed_zs_all = (
    observed_ave_df.groupby(group_keys + ["idx"]).mean()[signal_key] - shuffle_mu
) / shuffle_sig
observed_zs_all = (
    observed_zs_all.reset_index().set_index(group_keys + ["idx"]).sort_index()
)
observed_zs_average = observed_zs_all.groupby(["x", "init_type", "idx"]).mean()

one_sec_samples = int(fs)
vals = (
    observed_zs_all.loc[-one_sec_samples:one_sec_samples]
    .groupby(["init_type", "mouse_id"])
    .apply(lambda x: x.abs().mean())
    .dropna()
)

# ---------------------------------------------------------------------
# Wilcoxon test: movement vs syllable initiation (as in original)
# ---------------------------------------------------------------------
use_test = wilcoxon
stats = {}
stats["movement-syllables_offline"] = {}
stats["movement-syllables_offline"]["test"] = str(use_test.__name__)
stat, p = use_test(
    vals.loc["movement"][signal_key].values,
    vals.loc["syllables_offline"][signal_key].values,
)
stats["movement-syllables_offline"]["stat"] = stat
stats["movement-syllables_offline"]["p"] = p
stats["movement-syllables_offline"]["n1"] = len(vals.loc["movement"])
stats["movement-syllables_offline"]["n2"] = len(vals.loc["syllables_offline"])
stats["movement-syllables_offline"]["n_definition"] = "nmice"
stats["movement-syllables_offline"]["n_snippets_movement"] = snippet_df[
    "snippet"
].nunique()
stats

# save stats
if "dlight" in proc_dirs:
    os.makedirs(proc_dirs["dlight"], exist_ok=True)
    stats_path = os.path.join(
        proc_dirs["dlight"],
        f"stats_movement_init_{use_area}_renorm-{renormalize}.toml",
    )
    with open(stats_path, "w") as f:
        toml.dump(stats, f)
    print("Saved stats to", stats_path)

# ---------------------------------------------------------------------
# Plot panel: observed z with shuffle CI band (Fig. 1f-style)
# ---------------------------------------------------------------------
setup_plotting_env()

observed_cis = (
    (
        (observed_ave_df.groupby(group_keys + ["idx"])[signal_key].mean() - shuffle_mu)
        / shuffle_sig
    )
    .groupby(group_keys)
    .quantile([0.025, 0.975])
).dropna()
observed_cis.index = observed_cis.index.set_names("ci", level=-1)
observed_cis = observed_cis.groupby(["x", "init_type", "ci"]).mean()

observed_z_mean = observed_zs_average.groupby(["x", "init_type"]).mean()

# colors as in original
colors = [sns.color_palette()[1], geno_palette.snc]
plt_types = ["movement", "syllables_offline"]

fig, ax = plt.subplots(1, figsize=(1, 1.5))

for _type, _color in zip(plt_types, colors):
    obs_lo = observed_cis.xs(_type, level="init_type").iloc[::2]
    obs_hi = observed_cis.xs(_type, level="init_type").iloc[1::2]
    plt_xvec = obs_lo.index.get_level_values("x") / fs  # samples → s

    ax.fill_between(
        plt_xvec,
        obs_lo.values,
        obs_hi.values,
        fc=_color,
        alpha=0.5,
        zorder=100,
        lw=0,
        ec="none",
    )
    ax.plot(
        plt_xvec,
        observed_z_mean.xs(_type, level="init_type").values,
        color=_color,
        lw=1,
    )

    ax.set_xlim(-1.5, 1.5)

clean_ticks(ax, "y", include_zero=True, zero_symmetry=True, precision=0, dtype=int)
clean_ticks(ax, "x", precision=1, dtype=float)
ax.axvline(x=0, color=plt.rcParams["axes.edgecolor"], zorder=-150, alpha=1, ls="--")

ax.fill_between(
    ax.get_xlim(), lims.values[0], lims.values[1], fc=[0.75] * 3, alpha=1, zorder=-200
)

plt.ylabel("dF/F0 (z)")
plt.xlabel("Time (s)")
sns.despine()
plt.show()

# save panel
if "store_dir" in figure_cfg:
    os.makedirs(figure_cfg["store_dir"], exist_ok=True)
    fig_path = os.path.join(
        figure_cfg["store_dir"],
        f"dlight_initiation_grand_average_quartiles_merged_{use_area}_renorm-{renormalize}",
    )
    savefig(fig, fig_path, bbox_inches="tight")
    print("Saved figure to", fig_path)

# ---------------------------------------------------------------------
# Store full z-scored curves per mouse/snippet for later use
# ---------------------------------------------------------------------
use_group_keys = list(set(group_keys).union(set(["mouse_id", "idx"])))
observed_zs_all_store = (
    observed_ave_df.groupby(use_group_keys).mean()[signal_key] - shuffle_mu
) / shuffle_sig
observed_zs_all_store = observed_zs_all_store.dropna()

if "dlight" in proc_dirs:
    out_path = os.path.join(
        proc_dirs["dlight"],
        f"snippet_max_bin_ave_zscored_{use_area}_renorm-{renormalize}.parquet",
    )
    observed_zs_all_store.reset_index().to_parquet(out_path)
    print("Saved z-scored snippet averages to", out_path)

# ---------------------------------------------------------------------
# Randomization with respect to scalar values (bottom part of Fig. 1f panel)
# ---------------------------------------------------------------------
for (_plt_scalar_bin, _plt_dlight_bin, _plt_scalar) in tqdm(
    zip(plt_scalar_bins, plt_dlight_bins, plt_scalars)
):
    snippet_lst = snippet_df["snippet"].unique()
    shuffle_quantity = pd.Series(use_df["snippet"].unique()).rename("snippet")

    func = partial(
        get_shuffle_average,
        use_df=use_df[
            [_plt_scalar_bin, _plt_dlight_bin, use_feature, "x", "init_type", "snippet"]
        ],
        shuffle_quantity=None,
        shuffle_x=True,
        dlight_bin=["x", "init_type", _plt_scalar_bin, _plt_dlight_bin],
    )

    delays = [delayed(func)(sidx) for sidx in range(nshuffles)]
    results = Parallel(n_jobs=n_jobs, verbose=10, backend="multiprocessing")(delays)

    shuffle_grand_ave_df = pd.concat(results, axis=0)

    group_keys = ["x", "init_type", _plt_dlight_bin, _plt_scalar_bin]
    signal_key = use_feature
    observed_mean = snippet_df.groupby(group_keys)[signal_key].mean()
    shuffle_mu = shuffle_grand_ave_df.groupby(group_keys)[signal_key].mean()
    shuffle_sig = shuffle_grand_ave_df.groupby(group_keys)[signal_key].std()

    shuffle_grand_ave_df = shuffle_grand_ave_df.reset_index()

    _test = (
        shuffle_grand_ave_df.set_index(group_keys)[signal_key] - shuffle_mu
    ) / shuffle_sig
    shuffle_cis = _test.groupby(group_keys).quantile([0.025, 0.975])
    shuffle_cis.index = shuffle_cis.index.set_names("ci", level=-1)

    lims = shuffle_cis.groupby("ci").mean().values

    observed_z = (observed_mean - shuffle_mu) / shuffle_sig
    observed_z = observed_z.reset_index()
    observed_z["x"] /= fs  # samples → seconds

    show_scalar_bins = observed_z[_plt_scalar_bin].dropna().sort_values().unique()

    pal = sns.color_palette("Greens", n_colors=len(show_scalar_bins) + 1)[1:]
    cmap = plt.matplotlib.colors.LinearSegmentedColormap.from_list(
        "",
        pal,
        len(show_scalar_bins),
    )

    sm = plt.matplotlib.cm.ScalarMappable(
        cmap=cmap, norm=plt.Normalize(vmin=-0.1, vmax=1.1)
    )
    sm.set_array([])

    setup_plotting_env()

    g = sns.relplot(
        data=observed_z.loc[
            (observed_z["init_type"] == "syllables_offline")
            & (observed_z["x"].between(-1.5, 1.5, inclusive="both"))
        ].reset_index(),
        x="x",
        y=signal_key,
        row=_plt_scalar_bin,
        row_order=reversed(show_scalar_bins),
        hue=_plt_dlight_bin,
        ci=None,
        height=0.7,
        aspect=2.25,
        n_boot=0,
        kind="line",
        legend=False,
        palette=cmap,
        lw=1,
        clip_on=False,
    )
    g.fig.set_size_inches(1.5, 3)
    g.set_titles(row_template="{row_name}")
    g.set(xlim=(-1.5, 1.5), ylim=(-400, 400))

    yscale = 100
    yloc = -300
    xscale = 0.5
    xloc = -3

    for _ax in g.axes.flat:
        title = _ax.get_title()
        _ax.set_title("")
        _ax.set_title(title, loc="right", y=1, pad=-10)
        _ax.axis("off")
        _ax.axhline(y=0, color="k", zorder=-300, alpha=0.5)

    ylims = _ax.get_ylim()
    total_span = (ylims[1] - ylims[0]) * len(g.axes.flat)
    _ax.plot(
        [0, 0],
        [ylims[0], total_span * 0.9],
        color="k",
        zorder=-200,
        clip_on=False,
        alpha=0.5,
    )
    _ax.set_ylim(ylims)
    g.axes.flat[-1].plot([xloc, xloc], [yloc, yloc + yscale], color="k", clip_on=False)
    g.axes.flat[-1].plot([xloc, xloc + xscale], [yloc, yloc], color="k", clip_on=False)

    g.fig.subplots_adjust(hspace=0)
    cb = g.fig.colorbar(
        sm,
        ax=g.axes.ravel().tolist(),
        pad=0.3,
        label="Max dF/F0 bin",
        shrink=0.5,
        ticks=np.linspace(0, 1, len(show_scalar_bins)),
    )
    cb.set_ticklabels(show_scalar_bins)

    if "store_dir" in figure_cfg:
        out_path = os.path.join(
            figure_cfg["store_dir"],
            f"dlight_speed_transition_bins_max_split_shufflez_{_plt_scalar}_{use_area}_renorm-{renormalize}",
        )
        savefig(g.fig, out_path, bbox_inches="tight")
        print("Saved scalar-z plot to", out_path)

    observed_scalar_mean = (
        snippet_df.groupby(group_keys)[_plt_scalar].mean().reset_index()
    )

    pal = sns.color_palette("Blues", n_colors=len(show_scalar_bins) + 1)[1:]
    cmap = plt.matplotlib.colors.LinearSegmentedColormap.from_list(
        "",
        pal,
        len(show_scalar_bins),
    )

    sm = plt.matplotlib.cm.ScalarMappable(
        cmap=cmap, norm=plt.Normalize(vmin=-0.1, vmax=1.1)
    )
    sm.set_array([])

    observed_scalar_mean["x"] /= fs

    g = sns.relplot(
        data=observed_scalar_mean.loc[
            (observed_scalar_mean["init_type"] == "syllables_offline")
            & (observed_scalar_mean["x"].between(-1.5, 1.5, inclusive="both"))
        ],
        x="x",
        y=_plt_scalar,
        row=_plt_scalar_bin,
        row_order=reversed(show_scalar_bins),
        hue=_plt_dlight_bin,
        ci=None,
        height=0.7,
        aspect=2.25,
        n_boot=0,
        kind="line",
        legend=False,
        palette=cmap,
        lw=1,
        facet_kws={"sharey": True, "sharex": True},
    )
    g.set_titles(row_template="{row_name}")
    g.fig.set_size_inches(1.5, 3)

    yscale = 0.5
    yloc = -1
    xscale = 0.5
    xloc = -3

    for _ax in g.axes.flat:
        title = _ax.get_title()
        _ax.set_title("")
        _ax.set_title(title, loc="right", y=1, pad=-10)
        _ax.axis("off")
        _ax.axhline(y=0, color="k", zorder=-300, alpha=0.5)

    g.set(xlim=(-1.5, 1.5), ylim=(-1.5, 1.5))
    ylims = _ax.get_ylim()

    yrange = ylims[1] - ylims[0]
    _ax.plot(
        [0, 0], [ylims[0], yrange * 4], color="k", zorder=-200, clip_on=False, alpha=0.5
    )
    _ax.set_ylim(ylims)

    g.axes.flat[-1].plot([xloc, xloc], [yloc, yloc + yscale], color="k", clip_on=False)
    g.axes.flat[-1].plot([xloc, xloc + xscale], [yloc, yloc], color="k", clip_on=False)

    g.fig.subplots_adjust(hspace=0)
    cb = g.fig.colorbar(
        sm,
        ax=g.axes.ravel().tolist(),
        pad=0.3,
        label="Max dF/F0 bin",
        shrink=0.5,
        ticks=np.linspace(0, 1, len(show_scalar_bins)),
    )
    cb.set_ticklabels(show_scalar_bins)

    if "store_dir" in figure_cfg:
        out_path = os.path.join(
            figure_cfg["store_dir"],
            f"dlight_speed_transition_bins_max_split_{_plt_scalar}_{use_area}_renorm-{renormalize}",
        )
        savefig(g.fig, out_path, bbox_inches="tight")
        print("Saved scalar-mean plot to", out_path)
